In [14]:
from __future__ import absolute_import, division, print_function

import tensorflow as tf
from tensorflow.keras import Model, layers
import numpy as np

# Hyperparameters
num_classes = 10          # MNIST digit classes: 0-9
num_features = 784        # 28x28 images flattened

learning_rate = 0.1
training_steps = 2000
batch_size = 256
display_step = 100

n_hidden_1 = 128          #  hidden layer
n_hidden_2 = 256          # second hidden layer


In [15]:
# prepare MNIST
from tensorflow.keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# cast to float32
x_train = np.array(x_train, np.float32)
x_test  = np.array(x_test,  np.float32)

# flatten 28x28 -> 784
x_train = x_train.reshape([-1, num_features])
x_test  = x_test.reshape([-1, num_features])

# normalize
x_train = x_train / 255.
x_test  = x_test  / 255.

# tf.data pipeline
train_data = (
    tf.data.Dataset
    .from_tensor_slices((x_train, y_train))
    .repeat()
    .shuffle(5000)
    .batch(batch_size)
    .prefetch(1)
)

In [16]:
# Define Model
class NeuralNet(Model):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.fc1 = layers.Dense(n_hidden_1, activation=tf.nn.relu)
        self.fc2 = layers.Dense(n_hidden_2, activation=tf.nn.relu)
        self.out = layers.Dense(num_classes)  # logits

    def call(self, x, is_training=False):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.out(x)  # logits
        if not is_training:
            x = tf.nn.softmax(x)
        return x

neural_net = NeuralNet()

In [17]:
# Loss function
def cross_entropy_loss(logits, y_true):
    # y_true: int labels shape [batch]
    y_true = tf.cast(y_true, tf.int64)
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=y_true,
        logits=logits
    )
    return tf.reduce_mean(loss)


In [18]:
# Accuracy metric
def accuracy(y_pred, y_true):
    pred_classes = tf.argmax(y_pred, axis=1)
    y_true = tf.cast(y_true, tf.int64)
    correct = tf.equal(pred_classes, y_true)
    return tf.reduce_mean(tf.cast(correct, tf.float32))


In [19]:
# Optimizer
optimizer = tf.optimizers.SGD(learning_rate=learning_rate)


In [20]:
# Single optimization step
@tf.function
def run_optimization(x, y):
    with tf.GradientTape() as g:
        logits = neural_net(x, is_training=True)
        loss = cross_entropy_loss(logits, y)

    # compute gradients
    trainable_vars = neural_net.trainable_variables
    gradients = g.gradient(loss, trainable_vars)

    # update params
    optimizer.apply_gradients(zip(gradients, trainable_vars))

    return loss

In [21]:
# Training loop
train_data_iter = iter(train_data)

for step in range(1, training_steps + 1):
    # get next batch
    batch_x, batch_y = next(train_data_iter)

    # do one optimization step, get loss
    loss_value = run_optimization(batch_x, batch_y)

    # log every display_step
    if step % display_step == 0 or step == 1:
        # for accuracy we want predictions with softmax (is_training=False)
        preds = neural_net(batch_x, is_training=False)
        acc = accuracy(preds, batch_y)

        print(
            f"step: {step:4d}  "
            f"loss: {loss_value.numpy():.4f}  "
            f"acc: {acc.numpy():.4f}"
        )

step:    1  loss: 2.3093  acc: 0.1914
step:  100  loss: 0.4210  acc: 0.9023
step:  200  loss: 0.4271  acc: 0.9102
step:  300  loss: 0.2903  acc: 0.9375
step:  400  loss: 0.2906  acc: 0.9258
step:  500  loss: 0.2360  acc: 0.9414
step:  600  loss: 0.1899  acc: 0.9570
step:  700  loss: 0.2179  acc: 0.9688
step:  800  loss: 0.2331  acc: 0.9492
step:  900  loss: 0.2751  acc: 0.9414
step: 1000  loss: 0.2409  acc: 0.9414
step: 1100  loss: 0.1849  acc: 0.9531
step: 1200  loss: 0.2009  acc: 0.9688
step: 1300  loss: 0.1752  acc: 0.9609
step: 1400  loss: 0.1968  acc: 0.9375
step: 1500  loss: 0.1996  acc: 0.9609
step: 1600  loss: 0.1604  acc: 0.9688
step: 1700  loss: 0.1071  acc: 0.9805
step: 1800  loss: 0.0890  acc: 0.9883
step: 1900  loss: 0.0746  acc: 0.9766
step: 2000  loss: 0.0805  acc: 0.9805


In [22]:
# Evaluation on test set
# model outputs softmax if is_training=False
test_preds = neural_net(x_test, is_training=False)
test_acc = accuracy(test_preds, y_test)

print(f"Test accuracy: {test_acc.numpy():.4f}")

Test accuracy: 0.9627


batch al


model(batch) → logits


loss(logits, labels)


gradient(loss, model_weights)


optimizer.apply_gradients(...)


ağırlıklar güncellendi → model biraz daha iyi oldu


döngüye geri dön
